In [ ]:
%pip install arxiv

In [ ]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import arxiv
import pandas as pd

In [ ]:
import os, time, re, arxiv, pandas as pd, requests

os.makedirs("papers", exist_ok=True)

client = arxiv.Client(delay_seconds=3, num_retries=3)

search = arxiv.Search(
    query="cat:cs.AI",
    max_results=20,
    sort_by=arxiv.SortCriterion.SubmittedDate,
)

papers = []
for result in client.results(search):
    short_id = result.get_short_id()
    path = f"papers/{short_id}.pdf"

    papers.append({
        "arxiv_id": short_id,
        "title": re.sub(r"\s+", " ", result.title).strip(),
        "abstract": re.sub(r"\s+", " ", result.summary).strip(),
        "authors": [a.name for a in result.authors],
        "primary_category": result.primary_category,
        "categories": result.categories,
        "published": result.published,
        "updated": result.updated,
        "pdf_url": result.pdf_url,
        "abs_url": result.entry_id,
        "pdf_path": path,
    })
if not os.path.exists(path):          # skip re-downloads on rerun
        url = result.pdf_url.replace("://arxiv.org", "://export.arxiv.org")
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        with open(path, "wb") as f:
            f.write(r.content)
        print("Downloaded:", short_id)
        time.sleep(3)

df = pd.DataFrame(papers)